# 履歷資料寫入 Supabase

從 `resume.csv` 讀取測試履歷，使用 `supabase_connection.connect_to_supabase()` 連線，寫入 `resume` 資料表。

**執行條件**：
- 工作目錄：`supabase_control`（確保 `resume.csv`、`.env` 路徑正確）
- `USER` 表需有 user_id 1～10
- `resume_template` 表需有 template_id 1

## 1. 連線 Supabase

In [1]:
import sys
import os
from pathlib import Path

# 確保可 import 同資料夾的 supabase_connection（Notebook 請在 supabase_control 下開啟）
cwd = Path.cwd()
if str(cwd) not in sys.path:
    sys.path.insert(0, str(cwd))

from supabase_connection import connect_to_supabase

supabase = connect_to_supabase()

✓ Supabase 連線成功！資料庫中現有 1 筆公司資料（測試查詢）


## 2. 讀取 resume.csv 並轉換格式

In [2]:
import pandas as pd
import json

csv_path = Path("resume.csv")
if not csv_path.exists():
    csv_path = Path.cwd() / "resume.csv"

df = pd.read_csv(csv_path)
df = df.dropna(how="all")
print(f"✓ 載入 {len(df)} 筆履歷資料")

✓ 載入 10 筆履歷資料


In [3]:
def row_to_payload(row):
    """將 CSV 一列轉成 resume 表所需格式"""
    # JSON 欄位：字串 -> dict
    structured = row["structured_data"]
    normalized = row["normalized_data"]
    if isinstance(structured, str):
        try:
            structured = json.loads(structured)
        except json.JSONDecodeError:
            structured = None
    if isinstance(normalized, str):
        try:
            normalized = json.loads(normalized)
        except json.JSONDecodeError:
            normalized = None

    # 布林值
    def to_bool(v):
        if v is None or (isinstance(v, float) and pd.isna(v)):
            return False
        if isinstance(v, bool):
            return v
        return str(v).strip().upper() == "TRUE"

    # vector_id：空字串 / NaN -> None
    vid = row.get("vector_id")
    if pd.isna(vid) or (isinstance(vid, str) and not vid.strip()):
        vid = None
    else:
        vid = str(vid).strip()

    payload = {
        "resume_id": int(row["resume_id"]),
        "user_id": int(row["user_id"]),
        "template_id": int(row["template_id"]),
        "resume_type": str(row["resume_type"]),
        "structured_data": structured,
        "normalized_data": normalized,
        "vector_id": vid,
        "is_embedded": to_bool(row.get("is_embedded")),
        "is_primary": to_bool(row.get("is_primary")),
        "created_at": str(row["created_at"]) if pd.notna(row.get("created_at")) else None,
        "updated_at": str(row["updated_at"]) if pd.notna(row.get("updated_at")) else None,
    }
    return payload

rows = [row_to_payload(r) for _, r in df.iterrows()]
print(f"✓ 轉換 {len(rows)} 筆 payload")

✓ 轉換 10 筆 payload


## 3. 寫入 resume 表（Upsert）

In [4]:
success = 0
failed = 0

for r in rows:
    try:
        res = supabase.table("resume").upsert(
            r,
            on_conflict="resume_id",
            ignore_duplicates=False
        ).execute()
        if res.data:
            success += len(res.data)
        else:
            success += 1
    except Exception as e:
        print(f"❌ resume_id={r['resume_id']}: {e}")
        failed += 1

print(f"--- 寫入結果 ---")
print(f"✓ 成功: {success} 筆")
print(f"❌ 失敗: {failed} 筆")

--- 寫入結果 ---
✓ 成功: 10 筆
❌ 失敗: 0 筆


## 4. 驗證寫入結果

In [ ]:
verify = supabase.table("resume").select("resume_id", "user_id", "resume_type", count="exact").limit(20).execute()
total = getattr(verify, "count", None) or len(verify.data)
print(f"📊 資料庫 resume 表總數: {total} 筆")
display(pd.DataFrame(verify.data))

📊 資料庫 resume 表總數: 10 筆


,resume_id,user_id,resume_type
0,1,1,generated
1,2,2,generated
2,3,3,generated
3,4,4,generated
4,5,5,generated
5,6,6,generated
6,7,7,generated
7,8,8,generated
8,9,9,generated
9,10,10,generated


: 